# FastText analysis

Train FastText on the raw training tweets only, export vocabulary metadata, and inspect nearest neighbors.

In [1]:
import os
os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')
os.environ.setdefault('OMP_NUM_THREADS', '1')
os.environ.setdefault('MKL_NUM_THREADS', '1')

from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'src').is_dir())
sys.path.insert(0, str(PROJECT_ROOT))

from src.features import train_fasttext, export_vocab, nearest_neighbors
import json
from src.preprocess import PreprocessConfig, Preprocessor

DATA_DIR = PROJECT_ROOT / 'data'
train = pd.read_csv(DATA_DIR / 'train.tsv', sep='\t', header=None, names=['uid', 'text', 'label'])
train['text'] = train['text'].fillna('').astype(str)
preprocessor = Preprocessor(PreprocessConfig()).fit(train['text'])
train_token_sequences = preprocessor.transform(train['text'])

embedding_dir = PROJECT_ROOT / 'embeddings'
model_path = train_fasttext(train_token_sequences, embedding_dir / 'fasttext.model')
export_vocab(model_path, embedding_dir / 'vocab.json')
neighbors = nearest_neighbors(model_path, 'good', topn=10)
(embedding_dir / 'nearest_neighbors.json').write_text(json.dumps(neighbors, indent=2), encoding='utf-8')
neighbors


[('goodmorn', 0.9504114389419556),
 ('goodd', 0.9471163749694824),
 ('morn', 0.9219919443130493),
 ('goodluck', 0.9174478054046631),
 ('goodnight', 0.911589503288269),
 ('sebtoogood', 0.9114051461219788),
 ('gooby', 0.9073135852813721),
 ('afternoon', 0.8922652006149292),
 ('night', 0.8921374082565308),
 ('goodmnight', 0.8919380903244019)]